# C 与 CPython：代码和实验结果

[阅读正文](README.md) · [返回仓库首页](../../README.md)

实验分为两部分：保存 C 的预处理、编译、汇编和链接结果；分开观察 CPython 的编译、字节码与执行。

代码单元格下的输出整理自仓库已有的 Linux 验证记录（2026-10-11 10:33:48 UTC）。本次迁移没有重新执行实验，执行编号留空；这些输出不是当前电脑的新运行结果。重新运行全部单元格时，输出会被新环境的实际结果替换。

| 原记录环境 | 版本 |
| --- | --- |
| 操作系统与架构 | Linux x86_64 |
| Python | CPython 3.13.5 |
| C 编译器 | GCC 14.2.0，Debian 14.2.0-19 |

在 GitHub 上可以直接阅读代码和已有输出。复现需要 Linux、Python 3、GCC、GNU Binutils 和 `file`；在 Jupyter 中选择 Python 3 内核，再从上到下运行。源文件与构建产物写入临时目录，章节目录不需要额外的 `.c` 或 `.py` 文件。

## 运行环境

复现时先检查平台和工具，再创建临时实验目录。

In [ ]:
import contextlib
import dis
import io
from pathlib import Path
import platform
import shutil
import subprocess
import tempfile

if platform.system() != "Linux":
    raise RuntimeError("本实验使用 Linux、GCC 与 GNU Binutils。")

missing = [name for name in ("gcc", "file", "nm", "objdump") if not shutil.which(name)]
if missing:
    raise RuntimeError("缺少工具：" + ", ".join(missing))

compiler_version = subprocess.run(
    ["gcc", "--version"], capture_output=True, text=True, check=True, timeout=30
).stdout.splitlines()[0]
print("操作系统：", platform.system(), platform.machine())
print("Python：", platform.python_implementation(), platform.python_version())
print("编译器：", compiler_version)

temporary_directory = tempfile.TemporaryDirectory(prefix="program-execution-")
work_dir = Path(temporary_directory.name)

def run_command(arguments, expect_success=True):
    result = subprocess.run(
        arguments, cwd=work_dir, capture_output=True, text=True, timeout=30
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if expect_success:
        result.check_returncode()
    return result

操作系统： Linux x86_64
Python： CPython 3.13.5
编译器： gcc (Debian 14.2.0-19) 14.2.0


## 实验一：C 的构建与执行

`main.c` 调用 `add()`；`add.c` 提供它的实现。两个源文件分别生成目标文件，最后再链接在一起。

### 源文件 `main.c`

In [ ]:
main_c = r"""#include <stdio.h>

int add(int a, int b);

int main(void) {
    printf("%d\n", add(2, 3));
    return 0;
}
"""
_ = (work_dir / "main.c").write_text(main_c, encoding="utf-8")

### 源文件 `add.c`

In [ ]:
add_c = r"""int add(int a, int b) {
    return a + b;
}
"""
_ = (work_dir / "add.c").write_text(add_c, encoding="utf-8")

### 预处理

`-E` 停在预处理之后，将结果保存为 `main.i`；指定输出文件时，成功执行不会在终端打印正文。

In [ ]:
_ = run_command(["gcc", "-E", "main.c", "-o", "main.i"])

### 编译

`-S` 生成汇编文本 `main.s`，`-O0` 暂时关闭优化。

In [ ]:
_ = run_command(["gcc", "-S", "-O0", "main.i", "-o", "main.s"])

### 汇编

将 `main.s` 转成目标文件 `main.o`。

In [ ]:
_ = run_command(["gcc", "-c", "main.s", "-o", "main.o"])

### 为 `add.c` 生成目标文件

`gcc -c add.c` 仍会经过预处理与编译，只是在生成目标文件后停止，不进行链接。

In [ ]:
_ = run_command(["gcc", "-c", "-O0", "add.c", "-o", "add.o"])

### 链接并运行

把两个目标文件链接成 `demo`，然后执行。

In [ ]:
_ = run_command(["gcc", "main.o", "add.o", "-o", "demo"])

In [ ]:
demo = run_command(["./demo"])
assert demo.stdout == "5\n"

5


### 查看文件类型

`main.i` 与 `main.s` 是文本；本次 Linux x86_64 记录中的 `main.o` 是 ELF 可重定位目标文件，`demo` 是 ELF 可执行程序。

In [ ]:
_ = run_command(["file", "main.i", "main.s", "main.o", "demo"])

main.i: C source, ASCII text
main.s: assembler source, ASCII text
main.o: ELF 64-bit LSB relocatable, x86-64, version 1 (SYSV), not stripped
demo:   ELF 64-bit LSB pie executable, x86-64, version 1 (SYSV), dynamically linked, interpreter /lib64/ld-linux-x86-64.so.2, BuildID[sha1]=c55ab8bf4e668c59253a6dc28b96fc8cecccbd5a, for GNU/Linux 3.2.0, not stripped


### 查看跨文件的符号引用

在 GNU `nm` 的输出中，`U add` 表示 `main.o` 引用了尚未在该目标文件中定义的 `add`；`T add` 表示 `add.o` 中定义了它。

In [ ]:
main_symbols = run_command(["nm", "main.o"])
assert " U add" in main_symbols.stdout

                 U add
0000000000000000 T main
                 U printf


In [ ]:
add_symbols = run_command(["nm", "add.o"])
assert " T add" in add_symbols.stdout

0000000000000000 T add


### 查看目标文件的机器码反汇编

中间的十六进制字节是机器码，右侧是对应的汇编表示。这份输出属于 x86_64 架构，与后面 CPython 的字节码是不同的表示。

In [ ]:
_ = run_command(["objdump", "-d", "add.o"])


add.o:     file format elf64-x86-64


Disassembly of section .text:

0000000000000000 <add>:
   0:	55                   	push   %rbp
   1:	48 89 e5             	mov    %rsp,%rbp
   4:	89 7d fc             	mov    %edi,-0x4(%rbp)
   7:	89 75 f8             	mov    %esi,-0x8(%rbp)
   a:	8b 55 fc             	mov    -0x4(%rbp),%edx
   d:	8b 45 f8             	mov    -0x8(%rbp),%eax
  10:	01 d0                	add    %edx,%eax
  12:	5d                   	pop    %rbp
  13:	c3                   	ret


### 故意漏掉 `add.o`

这条命令预期失败，所以保留错误信息并检查非零退出码。`main.o` 引用了 `add`，但链接输入中缺少它的实现。

In [ ]:
failed_link = run_command(
    ["gcc", "main.o", "-o", "missing-add"], expect_success=False
)
print("退出码：", failed_link.returncode)
assert failed_link.returncode != 0
assert "add" in failed_link.stderr

/usr/bin/ld: main.o: in function `main':
main.c:(.text+0xf): undefined reference to `add'
collect2: error: ld returned 1 exit status
退出码： 1


源文件能分别编译，不意味着整个程序能够成功链接。缺少的定义要到链接阶段才能暴露出来。

## 实验二：CPython 的编译、字节码与执行

### 先运行对应的 Python 程序

下面是原来的 `main.py`，与 C 示例完成同样的加法和打印。

In [ ]:
"""Python counterpart to main.c + add.c."""


def add(a: int, b: int) -> int:
    return a + b


if __name__ == "__main__":
    print(add(2, 3))

5


### 只编译源代码

原来的 `inspect_python.py` 在这里拆成连续的三个单元格：编译、查看字节码、执行。

`compile()` 返回代码对象。参数 `"exec"` 表示输入是一组语句；源代码中的 `print()` 此时尚未执行。单独捕获编译步骤的标准输出，可以检查它是否为空。

In [ ]:
source = """
def add(a, b):
    return a + b

print(add(2, 3))
"""

print("1. 开始编译")
compile_stdout = io.StringIO()
with contextlib.redirect_stdout(compile_stdout):
    code_object = compile(source, "<example>", "exec")
print("2. 编译结束；source 内的 print 尚未执行")
print("compile() 的标准输出：", repr(compile_stdout.getvalue()))
assert compile_stdout.getvalue() == ""

1. 开始编译
2. 编译结束；source 内的 print 尚未执行
compile() 的标准输出： ''


### 查看代码对象的字节码

`dis.dis()` 显示 CPython 字节码，不会执行源代码里的加法和打印。以下字节码属于 CPython 3.13.5；指令名称、排列和代码对象地址可能随版本或运行变化。

`LOAD_CONST` 读取常量，`MAKE_FUNCTION` 创建函数对象，`CALL` 调用函数；在 `add` 的代码对象中，取参数、加法和返回分别由相应的字节码表示。这些字节码由 CPython 的执行机制处理。

In [ ]:
print("3. 查看字节码")
dis.dis(code_object)

3. 查看字节码
  0           RESUME                   0

  2           LOAD_CONST               0 (<code object add at 0x7f349954f4b0, file "<example>", line 2>)
              MAKE_FUNCTION
              STORE_NAME               0 (add)

  5           LOAD_NAME                1 (print)
              PUSH_NULL
              LOAD_NAME                0 (add)
              PUSH_NULL
              LOAD_CONST               1 (2)
              LOAD_CONST               2 (3)
              CALL                     2
              CALL                     1
              POP_TOP
              RETURN_CONST             3 (None)

Disassembly of <code object add at 0x7f349954f4b0, file "<example>", line 2>:
  2           RESUME                   0

  3           LOAD_FAST_LOAD_FAST      1 (a, b)
              BINARY_OP                0 (+)
              RETURN_VALUE


### 执行已经编译的代码对象

到 `exec()` 时，源代码中的 `print(add(2, 3))` 才真正执行。捕获的输出应严格等于 `5` 加一个换行。

In [ ]:
print("4. 开始执行 source")
exec_stdout = io.StringIO()
with contextlib.redirect_stdout(exec_stdout):
    exec(code_object, {})
print(exec_stdout.getvalue(), end="")
assert exec_stdout.getvalue() == "5\n"

4. 开始执行 source
5


### 清理临时构建文件

复现结束后删除临时目录。

In [ ]:
temporary_directory.cleanup()